In [12]:
import pandas as pd
import numpy as np


mobility = pd.read_csv("../data/mobility_va.csv")
mobility['date'] = pd.to_datetime(mobility['date'])

na_ratio = mobility.isna().mean()
print(na_ratio)

features = [
    'retail_and_recreation_percent_change_from_baseline', 
    'grocery_and_pharmacy_percent_change_from_baseline', 
    'parks_percent_change_from_baseline', 
    'transit_stations_percent_change_from_baseline', 
    'workplaces_percent_change_from_baseline', 
    'residential_percent_change_from_baseline'
]

rural_counties = ["Accomack County","Alleghany County","Bath County",
                  "Bland County","Brunswick County","Buchanan County",
                  "Carroll County","Charlotte County","Craig County",
                  "Dickenson County","Essex County","Grayson County",
                  "Greensville County","Halifax County","Henry County",
                  "Highland County","Lee County","Louisa County",
                  "Lunenburg County","Madison County","Mecklenburg County",
                  "Middlesex County","Montgomery County","Nelson County",
                  "Northampton County","Northumberland County","Patrick County",
                  "Pittsylvania County","Prince Edward County","Pulaski County",
                  "Richmond County","Rockbridge County","Rockingham County",
                  "Russell County","Smyth County","Southampton County",
                  "Tazewell County","Wise County","Wythe County","Shenandoah County"]

mobility['metro_label'] = np.where(mobility['sub_region_2'].isin(rural_counties), 0, 1)

Unnamed: 0                                            0.000000
country_region_code                                   0.000000
country_region                                        0.000000
sub_region_1                                          0.000000
sub_region_2                                          0.008328
metro_area                                            1.000000
iso_3166_2_code                                       0.991672
census_fips_code                                      0.008328
place_id                                              0.000000
date                                                  0.000000
retail_and_recreation_percent_change_from_baseline    0.366582
grocery_and_pharmacy_percent_change_from_baseline     0.402049
parks_percent_change_from_baseline                    0.844950
transit_stations_percent_change_from_baseline         0.645858
workplaces_percent_change_from_baseline               0.032072
residential_percent_change_from_baseline              0

/var/folders/lv/9m_qv53944x04_1mpk21jgtc0000gn/T/ipykernel_55836/1916626433.py:5: DtypeWarning: Columns (0: iso_3166_2_code) have mixed types. Specify dtype option on import or set low_memory=False.
  mobility = pd.read_csv("../data/mobility_va.csv")


In [13]:
features_keep = [f for f in features if mobility[f].notna().mean() > 0.3]

print("Keeping features:", features_keep)

Keeping features: ['retail_and_recreation_percent_change_from_baseline', 'grocery_and_pharmacy_percent_change_from_baseline', 'transit_stations_percent_change_from_baseline', 'workplaces_percent_change_from_baseline', 'residential_percent_change_from_baseline']


In [14]:
county_max_missing = (
    mobility
    .groupby('sub_region_2')[features_keep]
    .apply(lambda df: df.isna().mean().max())
)
#county_to_metro = mobility.groupby('sub_region_2')['metro_label'].first()
counties_keep = county_max_missing[county_max_missing < 0.5].index

# print(county_max_missing[county_max_missing < 0.3].size)
# thresh_5 = county_max_missing[county_max_missing < 0.5]
# thresh_7 = county_max_missing[county_max_missing < 0.7]
# print(county_max_missing[county_max_missing < 0.9].size)

# print(thresh_5[county_to_metro == 0].size)
# print(thresh_5[county_to_metro == 1].size)

# print(thresh_7[county_to_metro == 0].size)
# print(thresh_7[county_to_metro == 1].size)

mobility_counties_filtered = mobility[mobility['sub_region_2'].isin(counties_keep)]




In [16]:
mobility_clean = (
    mobility_counties_filtered
    .sort_values(['sub_region_2', 'date'])
    .groupby('sub_region_2', group_keys=False, as_index=False)
    .apply(lambda df: df[features_keep].interpolate(limit_direction='both')
                          .ffill()
                          .bfill()
                          .assign(metro_label=df['metro_label'], date=df['date'], sub_region_2=df['sub_region_2']))
)

KeyError: 'sub_region_2'

In [17]:
# Group and transform only the features, then join back/assign
mobility_clean = mobility_counties_filtered.sort_values(['sub_region_2', 'date']).copy()

# Fill missing values within each county group
mobility_clean[features_keep] = (
    mobility_clean
    .groupby('sub_region_2')[features_keep]
    .transform(lambda x: x.interpolate(limit_direction='both').ffill().bfill())
)

In [18]:
print(mobility_clean.shape)
print(mobility_clean['date'].min())
print(mobility_clean['date'].max())

(33013, 17)
2020-02-15 00:00:00
2022-10-15 00:00:00


In [20]:
monthly = pd.read_csv("../data/monthly.csv")
print(monthly.size)

14586


In [22]:
print(len(mobility_clean['sub_region_2'].unique()))
print(mobility_clean[:].isna().sum())

print(mobility_clean['metro_label'].sum()/len(mobility_clean))

mobility_clean.to_csv("data/mobility_clean.csv", index=False)

34
Unnamed: 0                                                0
country_region_code                                       0
country_region                                            0
sub_region_1                                              0
sub_region_2                                              0
metro_area                                            33013
iso_3166_2_code                                       33013
census_fips_code                                          0
place_id                                                  0
date                                                      0
retail_and_recreation_percent_change_from_baseline        0
grocery_and_pharmacy_percent_change_from_baseline         0
parks_percent_change_from_baseline                    17743
transit_stations_percent_change_from_baseline             0
workplaces_percent_change_from_baseline                   0
residential_percent_change_from_baseline                  0
metro_label                          

OSError: Cannot save file into a non-existent directory: 'data'

In [26]:
mobility_1 = mobility_clean.copy()
mobility_1["year_month"] = mobility_1["date"].dt.to_period('M')

monthly = mobility_1.groupby("year_month")[features_keep].mean()
monthly.to_csv("../data/monthly.csv")

In [27]:
monthly = pd.read_csv("../data/monthly.csv")
print(monthly.columns)

Index(['year_month', 'retail_and_recreation_percent_change_from_baseline',
       'grocery_and_pharmacy_percent_change_from_baseline',
       'transit_stations_percent_change_from_baseline',
       'workplaces_percent_change_from_baseline',
       'residential_percent_change_from_baseline'],
      dtype='str')
